In [ ]:
from kfp import dsl
from kfp import compiler
from kfp.dsl import Input, Output, Dataset, Model, Artifact


@dsl.component(
    base_image="python:3.11",
    packages_to_install=[
        "pandas",
        "numpy",
        "scikit-learn",
        "torch",
        "joblib"
    ]
)
def generate_data(
    output_csv: Output[Dataset]
):
    import numpy as np
    import pandas as pd

    np.random.seed(42)
    n = 1000

    wiek = np.random.randint(18, 70, n)
    dochod = np.random.randint(3000, 25000, n)
    wizyty_www = np.random.randint(1, 50, n)
    czas_na_stronie = np.random.uniform(0.5, 20.0, n)

    score = (
        0.02 * wiek
        + 0.00015 * dochod
        + 0.12 * wizyty_www
        + 0.35 * czas_na_stronie
        - 8
    )

    prob = 1 / (1 + np.exp(-score))
    kupi = (np.random.rand(n) < prob).astype(int)

    df = pd.DataFrame({
        "wiek": wiek,
        "dochod": dochod,
        "wizyty_www": wizyty_www,
        "czas_na_stronie": czas_na_stronie,
        "kupi": kupi
    })

    df.to_csv(output_csv.path, index=False)

    print("Zapisano dataset:", output_csv.path)
    print(df.head())


@dsl.component(
    base_image="python:3.11",
    packages_to_install=[
        "pandas",
        "numpy",
        "scikit-learn",
        "torch",
        "joblib"
    ]
)
def train_model(
    input_csv: Input[Dataset],
    model_artifact: Output[Model],
    scaler_artifact: Output[Artifact],
    metrics_artifact: Output[Artifact]
):
    import json
    import joblib
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn

    from torch.utils.data import TensorDataset, DataLoader
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

    RANDOM_STATE = 42
    BATCH_SIZE = 32
    EPOCHS = 20
    LEARNING_RATE = 0.001

    np.random.seed(RANDOM_STATE)
    torch.manual_seed(RANDOM_STATE)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    df = pd.read_csv(input_csv.path)
    print("Wczytano dane:", df.shape)

    required_columns = ["wiek", "dochod", "wizyty_www", "czas_na_stronie", "kupi"]
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"Brakuje kolumn: {missing}")

    feature_columns = ["wiek", "dochod", "wizyty_www", "czas_na_stronie"]
    target_column = "kupi"

    X = df[feature_columns].values
    y = df[target_column].values

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    class CustomerClassifier(nn.Module):
        def __init__(self, input_dim: int):
            super().__init__()
            self.network = nn.Sequential(
                nn.Linear(input_dim, 16),
                nn.ReLU(),
                nn.Linear(16, 8),
                nn.ReLU(),
                nn.Linear(8, 1)
            )

        def forward(self, x):
            return self.network(x)

    model = CustomerClassifier(input_dim=len(feature_columns)).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    train_losses = []

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)
        train_losses.append(epoch_loss)
        print(f"Epoch {epoch+1}/{EPOCHS}, loss={epoch_loss:.4f}")

    model.eval()

    all_preds = []
    all_true = []
    all_probs = []

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)

            logits = model(batch_x)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            all_probs.extend(probs.cpu().numpy().flatten().tolist())
            all_preds.extend(preds.cpu().numpy().flatten().tolist())
            all_true.extend(batch_y.numpy().flatten().tolist())

    accuracy = accuracy_score(all_true, all_preds)
    conf_matrix = confusion_matrix(all_true, all_preds).tolist()
    clf_report = classification_report(all_true, all_preds, output_dict=True)

    print("Accuracy:", accuracy)
    print("Confusion matrix:", conf_matrix)

    torch.save({
        "model_state_dict": model.state_dict(),
        "input_dim": len(feature_columns),
        "feature_columns": feature_columns
    }, model_artifact.path)

    joblib.dump(scaler, scaler_artifact.path)

    metrics = {
        "accuracy": float(accuracy),
        "confusion_matrix": conf_matrix,
        "classification_report": clf_report,
        "train_losses": train_losses
    }

    with open(metrics_artifact.path, "w") as f:
        json.dump(metrics, f, indent=2)

    print("Model zapisany do:", model_artifact.path)
    print("Scaler zapisany do:", scaler_artifact.path)
    print("Metrics zapisane do:", metrics_artifact.path)


@dsl.component(
    base_image="python:3.11",
    packages_to_install=["pandas", "torch", "joblib", "scikit-learn"]
)
def predict_sample(
    model_artifact: Input[Model],
    scaler_artifact: Input[Artifact]
):
    import joblib
    import pandas as pd
    import torch
    import torch.nn as nn

    class CustomerClassifier(nn.Module):
        def __init__(self, input_dim: int):
            super().__init__()
            self.network = nn.Sequential(
                nn.Linear(input_dim, 16),
                nn.ReLU(),
                nn.Linear(16, 8),
                nn.ReLU(),
                nn.Linear(8, 1)
            )

        def forward(self, x):
            return self.network(x)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    checkpoint = torch.load(model_artifact.path, map_location=device)
    scaler = joblib.load(scaler_artifact.path)

    model = CustomerClassifier(input_dim=checkpoint["input_dim"]).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    feature_columns = checkpoint["feature_columns"]

    new_customers = pd.DataFrame([
        {"wiek": 25, "dochod": 4500, "wizyty_www": 3, "czas_na_stronie": 1.2},
        {"wiek": 39, "dochod": 12000, "wizyty_www": 15, "czas_na_stronie": 8.5},
        {"wiek": 52, "dochod": 22000, "wizyty_www": 27, "czas_na_stronie": 14.0},
    ])

    X_new = new_customers[feature_columns].values
    X_new_scaled = scaler.transform(X_new)
    X_new_tensor = torch.tensor(X_new_scaled, dtype=torch.float32).to(device)

    with torch.no_grad():
        logits = model(X_new_tensor)
        probs = torch.sigmoid(logits).cpu().numpy().flatten()
        preds = (probs >= 0.5).astype(int)

    new_customers["prawdopodobienstwo_zakupu"] = probs
    new_customers["predykcja_kupi"] = preds

    print(new_customers.to_string(index=False))


@dsl.pipeline(
    name="pytorch-customer-classification",
    description="Prosty pipeline Kubeflow do treningu modelu PyTorch"
)
def customer_pipeline():
    data_task = generate_data()
    train_task = train_model(input_csv=data_task.outputs["output_csv"])
    predict_sample(
        model_artifact=train_task.outputs["model_artifact"],
        scaler_artifact=train_task.outputs["scaler_artifact"]
    )


if __name__ == "__main__":
    compiler.Compiler().compile(
        pipeline_func=customer_pipeline,
        package_path="pytorch_customer_pipeline.yaml"
    )